# 🎓 AlternIA Cloud GPU Server — Google Colab Pro

Bienvenue dans le notebook de déploiement cloud d'**AlternIA** !
Ce notebook permet d'exécuter toute la pile d'intelligence artificielle sur un GPU ultra-rapide (**NVIDIA A100 / L4 / T4**) :

1. 🧠 **LLM & RAG Pédagogique** : Qwen 2.5 7B/14B accéléré sur GPU + Recherche vectorielle dans les manuels scolaires maliens.
2. 🎬 **Générateur d'Avatar Vidéo 1-Click** : Transformation d'**une seule photo** en vidéo MP4 parlante ultra-réaliste synchronisée avec la voix (SadTalker / LivePortrait).
3. 🎙️ **Reconnaissance Vocale (STT)** : Faster-Whisper Large-v3 sur CUDA.
4. 🔊 **Synthèse Vocale (TTS)** : Neural Edge-TTS haute fidélité.
5. 🌐 **Tunnel Public Sécurisé** : Exposition de l'API en HTTPS (`trycloudflare.com`) pour connecter votre boîtier physique ou le web en direct.

## ⚡ Étape 1 : Vérification du GPU (NVIDIA A100 / L4 / T4)

In [ ]:
!nvidia-smi
import torch
print(f"✅ CUDA Disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU Actif : {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM Totale : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} Go")

## 📦 Étape 2 : Cloner le Répertoire & Installer les Dépendances

In [ ]:
# 1. Nettoyage et clonage du projet AlternIA
import os, sys, getpass
%cd /content
if os.path.exists("/content/AlternIA"):
    !rm -rf /content/AlternIA

repo_url = "https://github.com/skypper109/AlternIA.git"
res = os.system(f"git clone -b develops {repo_url} /content/AlternIA")
if res != 0:
    print("
⚠️ Le dépôt GitHub est privé ou inaccessible sans authentification.")
    token = getpass.getpass("🔑 Collez votre GitHub Personal Access Token (ou rendez le dépôt public) : ")
    if token:
        !git clone -b develops https://{token}@github.com/skypper109/AlternIA.git /content/AlternIA

%cd /content/AlternIA

# 2. Installer les packages système
!apt-get install -y -qq ffmpeg

# 3. Installer les dépendances Python du projet AlternIA
!pip install -q -r requirements.txt
!pip install -q faster-whisper ctranslate2 soundfile

# 4. Installer SadTalker (Générateur d'Avatar Vidéo GPU)
%cd /content
if not os.path.exists("/content/SadTalker"):
    !git clone https://github.com/OpenTalker/SadTalker.git /content/SadTalker

%cd /content/SadTalker
# Installation robuste pour éviter les erreurs de compilation wheel (basicsr/facexlib)
!pip install -q yacs face-alignment imageio[ffmpeg] resampy pydub kornia safetensors
!pip install -q --no-build-isolation basicsr facexlib gfpgan

# Télécharger les checkpoints pré-entraînés pour SadTalker
!mkdir -p checkpoints gfpgan/weights
!wget -q -nc -O checkpoints/SadTalker_V0.0.2_256.safetensors https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2/SadTalker_V0.0.2_256.safetensors
!wget -q -nc -O checkpoints/mapping_00109-model.pth.tar https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2/mapping_00109-model.pth.tar
!wget -q -nc -O checkpoints/mapping_00229-model.pth.tar https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2/mapping_00229-model.pth.tar
!wget -q -nc -O gfpgan/weights/alignment_WFLW_4HG.pth https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth
!wget -q -nc -O gfpgan/weights/detection_Resnet50_Final.pth https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth

%cd /content/AlternIA
print("
✅ Toutes les dépendances et modèles GPU sont prêts !")


## 🌐 Étape 3 : Lancer le Serveur AlternIA & Générer l'URL Publique HTTPS

In [ ]:
# Lancement du serveur FastAPI + Tunnel Cloudflare HTTPS
%cd /content/AlternIA
!python3 cloud/colab_server.py